
# 09 — Intelligent Request Routing

**Learning reference:** [BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Inspired by course notebook:** `routing.ipynb`

## Goal

Explore the routing architecture: classify an incoming
request and direct it to an appropriate workflow.

The course example routes requests to a spam classifier
or a text summarizer. This adaptation uses customer-support
intent classification.

The notebook covers:

- Rule-based routing using Python.
- Explicit route validation and fallback handling.
- Downstream task selection.
- Evaluating routing decisions using labeled examples.
- Comparing rule-based routing with live Gemini classification.

**API usage:** The offline section uses zero requests.
The optional Gemini section uses exactly one model request
per execution of its live classification cell, without
automatic retries.

The examples are synthetic. No real customer accounts
or external databases are accessed.



## 1. What Is Routing?

Routing is an architecture in which a request is
classified and sent to a suitable task or workflow.

For example:

- "Where is order 1001?" → Orders
- "Which headphones are available?" → Products
- "What is your return policy?" → Policies
- "Can you explain something unrelated?" → Clarification

The router's responsibility is to choose a destination.

The selected workflow is responsible for handling the
request. These are two separate responsibilities.

A router can use predefined Python rules or an LLM.

**Rule-based routing**
- Predictable and inexpensive.
- No API calls.
- Depends on explicitly defined matching rules.

**LLM routing**
- Can recognize different ways of expressing an intent.
- Requires a model request.
- May produce an incorrect or invalid classification.

We will implement both methods and compare their results.


In [1]:

import re
from collections import Counter

VALID_ROUTES = {
    "orders",
    "products",
    "policies",
    "clarify"
}

ROUTE_DESCRIPTIONS = {
    "orders": "Order status and tracking questions",
    "products": "Product catalogue and availability",
    "policies": "Returns, refunds and warranty rules",
    "clarify": "Unknown or multiple unrelated intents"
}

print("Available routes:")

for route, description in ROUTE_DESCRIPTIONS.items():
    print(f"- {route}: {description}")


Available routes:
- orders: Order status and tracking questions
- products: Product catalogue and availability
- policies: Returns, refunds and warranty rules
- clarify: Unknown or multiple unrelated intents



## 2. Rule-Based Router

Start with a simple deterministic router.

It looks for words associated with orders, products
and policies.

If exactly one intent is detected, the router selects it.

If no intent or multiple intents are detected, the
router returns `clarify` instead of guessing.

This is an intentionally simple keyword-based baseline.

It does not fully understand language or context.


In [2]:

INTENT_KEYWORDS = {
    "orders": [
        "order",
        "tracking",
        "track",
        "package"
    ],
    "products": [
        "product",
        "headphones",
        "earbuds",
        "speaker",
        "stock",
        "price",
        "available"
    ],
    "policies": [
        "return",
        "refund",
        "warranty",
        "policy"
    ]
}


def route_by_rules(message):
    if not isinstance(message, str):
        raise TypeError("Message must be a string.")

    text = message.lower()
    matches = set()

    for route, keywords in INTENT_KEYWORDS.items():
        for keyword in keywords:
            pattern = rf"\b{re.escape(keyword)}\b"

            if re.search(pattern, text):
                matches.add(route)
                break

    if len(matches) == 1:
        return next(iter(matches))

    return "clarify"


examples = [
    "Where is order 1001?",
    "Which headphones are available?",
    "What is your return policy?",
    "Tell me a joke",
    "Where is order 1001 and what is your return policy?"
]

for message in examples:
    print(f"{message} -> {route_by_rules(message)}")


Where is order 1001? -> orders
Which headphones are available? -> products
What is your return policy? -> policies
Tell me a joke -> clarify
Where is order 1001 and what is your return policy? -> clarify



## 3. Dispatching to Downstream Workflows

After classification, the application needs to choose
an appropriate downstream operation.

Examples:

- Orders → Look up order details.
- Products → Search the catalogue.
- Policies → Search policy documents.
- Clarify → Request a more specific question.

In this notebook, the handlers return structured
task instructions rather than performing live retrieval.

This keeps routing separate from the underlying business
logic, which was already explored in Notebook 06.


In [3]:

def handle_orders(message):
    return {
        "workflow": "order_lookup",
        "request": message,
        "status": "ready_for_order_service"
    }


def handle_products(message):
    return {
        "workflow": "catalogue_search",
        "request": message,
        "status": "ready_for_product_service"
    }


def handle_policies(message):
    return {
        "workflow": "policy_retrieval",
        "request": message,
        "status": "ready_for_policy_service"
    }


def handle_clarify(message):
    return {
        "workflow": "clarification",
        "request": message,
        "status": "needs_clarification"
    }


ROUTE_HANDLERS = {
    "orders": handle_orders,
    "products": handle_products,
    "policies": handle_policies,
    "clarify": handle_clarify
}


def dispatch_route(route, message):
    if route not in VALID_ROUTES:
        raise ValueError(f"Unsupported route: {route}")

    return ROUTE_HANDLERS[route](message)


question = "Where is order 1001?"

selected_route = route_by_rules(question)
result = dispatch_route(selected_route, question)

print("Selected route:", selected_route)
print("Downstream task:", result)


Selected route: orders
Downstream task: {'workflow': 'order_lookup', 'request': 'Where is order 1001?', 'status': 'ready_for_order_service'}



## 4. Routing Evaluation

A router should be tested against examples with
known expected classifications.

We will create a small labeled dataset.

For each example, compare:

- Expected route: The manually assigned correct label.
- Predicted route: The router's classification.

Then calculate classification accuracy.

This tests the rule-based router only. It does not
measure Gemini's performance.

The dataset is small and synthetic, so results should
not be interpreted as production accuracy.


In [4]:

test_cases = [
    ("Where is order 1001?", "orders"),
    ("Track my package", "orders"),
    ("Which headphones are available?", "products"),
    ("Show me the product price", "products"),
    ("What is the return policy?", "policies"),
    ("Do you offer a warranty?", "policies"),
    ("Tell me a joke", "clarify"),
    (
        "Where is order 1001 and what is your refund policy?",
        "clarify"
    )
]

evaluation_results = []

for message, expected in test_cases:
    predicted = route_by_rules(message)

    evaluation_results.append({
        "message": message,
        "expected": expected,
        "predicted": predicted,
        "correct": predicted == expected
    })

    print(
        f"Expected: {expected:9} "
        f"Predicted: {predicted:9} "
        f"Correct: {predicted == expected}"
    )

correct_count = sum(
    row["correct"]
    for row in evaluation_results
)

accuracy = correct_count / len(evaluation_results)

print(f"\nAccuracy: {accuracy:.1%}")
print(f"Correct: {correct_count}/{len(test_cases)}")


Expected: orders    Predicted: orders    Correct: True
Expected: orders    Predicted: orders    Correct: True
Expected: products  Predicted: products  Correct: True
Expected: products  Predicted: products  Correct: True
Expected: policies  Predicted: policies  Correct: True
Expected: policies  Predicted: policies  Correct: True
Expected: clarify   Predicted: clarify   Correct: True
Expected: clarify   Predicted: clarify   Correct: True

Accuracy: 100.0%
Correct: 8/8


In [5]:

assert route_by_rules("Where is order 1001?") == "orders"
assert route_by_rules("What is your return policy?") == "policies"

assert route_by_rules(
    "Order 1001 and return policy"
) == "clarify"

assert route_by_rules(
    "Hello there"
) == "clarify"

try:
    dispatch_route(
        "unknown_route",
        "Test message"
    )
    raise AssertionError("Unknown route was accepted.")
except ValueError:
    pass

assert set(ROUTE_HANDLERS) == VALID_ROUTES

print("All offline routing tests passed.")
print("Gemini API requests: 0")


All offline routing tests passed.
Gemini API requests: 0



## 5. LLM-Based Routing with Gemini

A keyword router depends on predefined vocabulary.

An LLM router can interpret the meaning of a message,
including wording that does not match our keyword list.

For example:

"Has my parcel been dispatched?"

This refers to order fulfillment, even though the
words `parcel` and `dispatched` are absent from the
rule-based router's keyword list.

We will ask Gemini to select exactly one destination:

- `orders`
- `products`
- `policies`
- `clarify`

If multiple categories apply or the request is unclear,
the correct destination is `clarify`.

The application validates Gemini's response before
using it as a route.

**API usage:** One Gemini request for one message.
No automatic retries or extra downstream model calls.


In [6]:

from openai import OpenAI
from getpass import getpass

MODEL = "gemini-3.5-flash-lite"

gemini_key = getpass("Enter your Gemini API key: ")

client = OpenAI(
    api_key=gemini_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    max_retries=0,
    timeout=120.0
)

safe_client = client.with_options(
    max_retries=0,
    timeout=120.0
)

print("Selected model:", MODEL)
print("Gemini client ready. No API request made.")


Enter your Gemini API key: ··········
Selected model: gemini-3.5-flash-lite
Gemini client ready. No API request made.


In [7]:

LIVE_MESSAGE = "Has my parcel been dispatched?"

rule_prediction = route_by_rules(LIVE_MESSAGE)

print("Customer message:", LIVE_MESSAGE)
print("Rule-based prediction:", rule_prediction)

response = safe_client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a customer-support intent router. "
                "Choose exactly one category: "
                "orders, products, policies, or clarify. "
                "Orders covers tracking, dispatch, and delivery status. "
                "Products covers catalogue, prices, and availability. "
                "Policies covers refunds, returns, and warranties. "
                "Choose clarify for unclear or multi-intent requests. "
                "Reply with only the category name in lowercase."
            )
        },
        {
            "role": "user",
            "content": LIVE_MESSAGE
        }
    ],
    temperature=0
)

raw_prediction = response.choices[0].message.content

if not raw_prediction:
    raise ValueError("Gemini returned an empty classification.")

gemini_prediction = raw_prediction.strip().lower()

if gemini_prediction not in VALID_ROUTES:
    raise ValueError(
        f"Invalid Gemini route: {gemini_prediction!r}"
    )

print("Gemini prediction:", gemini_prediction)

live_task = dispatch_route(
    gemini_prediction,
    LIVE_MESSAGE
)

print("Selected downstream task:", live_task)


Customer message: Has my parcel been dispatched?
Rule-based prediction: clarify
Gemini prediction: orders
Selected downstream task: {'workflow': 'order_lookup', 'request': 'Has my parcel been dispatched?', 'status': 'ready_for_order_service'}


In [8]:

expected_live_route = "orders"

print("Customer question:", LIVE_MESSAGE)
print("Expected route:", expected_live_route)
print("Rule-based route:", rule_prediction)
print("Gemini route:", gemini_prediction)

print("\nRule-based correct:",
      rule_prediction == expected_live_route)

print("Gemini correct:",
      gemini_prediction == expected_live_route)

print("\nValid Gemini route:",
      gemini_prediction in VALID_ROUTES)

print("Live classification API requests: 1")


Customer question: Has my parcel been dispatched?
Expected route: orders
Rule-based route: clarify
Gemini route: orders

Rule-based correct: False
Gemini correct: True

Valid Gemini route: True
Live classification API requests: 1



## Findings and Limitations

### Rule-Based Routing

- Implemented a deterministic intent classifier using keywords.
- Created separate destinations for orders, products and policies.
- Added a clarification route for unknown or multi-intent requests.
- Used an allowlisted dispatcher to select downstream workflows.
- Evaluated the router against eight synthetic examples.
- Confirmed that unsupported routes are rejected.

### LLM-Based Routing

- Used Gemini to classify a customer-support message.
- Validated the returned category against approved destinations.
- Used Python to select the corresponding downstream task.
- Limited the demonstration to one model request.

### Limitations

- The keyword rules cover only a small vocabulary.
- The offline evaluation examples are synthetic and limited.
- The LLM demonstration evaluates only one message.
- An LLM classification is not guaranteed to be correct.
- Invalid model outputs are rejected rather than automatically retried.
- Multi-intent requests are directed to clarification rather
  than decomposed into multiple workflows.
- Downstream handlers return task descriptions; they do not
  perform database or knowledge-base retrieval.
- No production authorization or customer identity checks
  are implemented.


### Observed Live Results

- The rule-based router returned `clarify` for
  "Has my parcel been dispatched?" because the relevant
  words were absent from its keyword list.
- Gemini 3.5 Flash Lite correctly classified the same
  message as `orders`.
- Python validated the classification and selected the
  `order_lookup` downstream workflow.
- The live comparison used one Gemini API request.
- This single example demonstrates semantic flexibility,
  but does not establish general classification accuracy.

### Conclusion

This notebook demonstrates routing as a separate
architectural responsibility: selecting the right
workflow before performing the requested task.

The rule-based approach provides predictable behavior
without API costs, while the LLM-based approach can
recognize more flexible language at the cost of
additional model requests.

Validation and fallback handling remain important
regardless of how routing decisions are made.
